# Validate Raw Data


In [1]:
# Local project setup
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

# Uncomment the next line in a fresh Colab runtime if dependencies are missing.
# %pip install -q -r requirements.txt scipy scikit-learn


Project root: g:\ACHINI\sliit activities\Y3S2\Data Science\FDM\FDM-Mini-Project


## Run the stage

Schema and data-quality validation for the EPA annual-summary dataset.

In [ ]:

from __future__ import annotations
from pathlib import Path
import json
import pandas as pd

REQUIRED_COLUMNS = {
    "state_code", "county_code", "site_num", "parameter_code", "poc",
    "latitude", "longitude", "parameter_name", "sample_duration",
    "pollutant_standard", "year", "units_of_measure", "observation_count",
    "observation_percent", "valid_day_count", "arithmetic_mean",
    "arithmetic_standard_dev", "first_max_value",
    "ninety_nine_percentile", "ninety_eight_percentile",
    "ninety_five_percentile", "ninety_percentile", "seventy_five_percentile",
    "fifty_percentile", "ten_percentile", "state_name", "county_name",
    "city_name"
}

def validate_schema(df: pd.DataFrame) -> None:
    missing = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing:
        raise ValueError(f"Missing required EPA columns: {missing}")

def quality_report(df: pd.DataFrame) -> dict:
    validate_schema(df)

    numeric_checks = {}
    for col in ["latitude", "longitude", "year", "observation_count",
                "observation_percent", "valid_day_count", "arithmetic_mean"]:
        s = pd.to_numeric(df[col], errors="coerce")
        numeric_checks[col] = {
            "non_numeric_or_missing": int(s.isna().sum()),
            "min": None if s.dropna().empty else float(s.min()),
            "max": None if s.dropna().empty else float(s.max()),
        }

    invalid = {
        "latitude_out_of_range": int((pd.to_numeric(df["latitude"], errors="coerce").abs() > 90).sum()),
        "longitude_out_of_range": int((pd.to_numeric(df["longitude"], errors="coerce").abs() > 180).sum()),
        "observation_percent_out_of_range": int(
            ((pd.to_numeric(df["observation_percent"], errors="coerce") < 0) |
             (pd.to_numeric(df["observation_percent"], errors="coerce") > 100)).sum()
        ),
        "negative_observation_count": int((pd.to_numeric(df["observation_count"], errors="coerce") < 0).sum()),
        "negative_valid_day_count": int((pd.to_numeric(df["valid_day_count"], errors="coerce") < 0).sum()),
    }

    return {
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "duplicate_full_rows": int(df.duplicated().sum()),
        "missing_by_column": {k: int(v) for k, v in df.isna().sum().items()},
        "numeric_checks": numeric_checks,
        "invalid_value_checks": invalid,
    }

def save_quality_report(df: pd.DataFrame, output_path: str | Path) -> dict:
    report = quality_report(df)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report

from src.data.load_data import load_air_quality


project_root = Path.cwd().parent
raw_csv = project_root / "data" / "raw" / "air_quality.csv"
report_path = project_root / "reports" / "data_quality_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)

data = load_air_quality(raw_csv)
report = save_quality_report(data, report_path)
print(json.dumps(report["invalid_value_checks"], indent=2))

{
  "latitude_out_of_range": 0,
  "longitude_out_of_range": 0,
  "observation_percent_out_of_range": 0,
  "negative_observation_count": 0,
  "negative_valid_day_count": 0
}
